# CFNA Large-Corpus + ForgeLoop Training Notebook

This notebook expands CFNA from a small TinyStories smoke test into a **balanced, provenance-recorded base corpus**, then performs a separate **response-only instruction-tuning stage**.

The coding curriculum is built around **CFNA ForgeLoop**, an original bounded software-engineering scaffold fitted to this repository:

`CONTRACT → MAP → PLAN → ACT → OBSERVE → CRITIQUE → REVISE → VERIFY → LEDGER → MEMORY`

ForgeLoop is not autonomous self-modification. It is a supervised training format: changes remain bounded by an authority contract, observations come from tools/tests, revisions are limited, and every result must pass verification before entering memory.

## Default corpus preset

The default preset targets roughly **0.8–1.0 GB**, depending on source availability:

- Cosmopedia-100k educational prose
- Permissively licensed Python/Rust/TypeScript/Shell code
- Public-domain books and U.S.-government material
- This Apache-2.0 CFNA repository
- Grounded ForgeLoop traces derived from the accepted code

TinyStories can be added with an explicit license-policy toggle. FineWeb/Common Crawl is intentionally excluded from the strict preset.

## Training stages

1. Build a real document-level train/validation split.
2. Continue base pretraining from the existing checkpoint.
3. Test raw story/code completion.
4. Build ForgeLoop + OpenAssistant instruction pairs.
5. Run response-only SFT and save a separate assistant checkpoint.

Large corpus size does **not** increase parameter count. The active PyTorch configuration remains approximately 11.1M parameters unless you deliberately change the architecture.


**Colab paid tier:** pick the fastest GPU under Runtime > Change runtime type (L4/A100 when available) + High-RAM. Batch size auto-scales to the GPU in the config cell. Pro+ background execution keeps this running after you close the tab; every stage is resume-safe regardless.


In [1]:
# 1) READY-TO-RUN SETUP FOR COLAB/JUPYTER

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/LMMinier/nueronce.git"
BRANCH = "claude/cfna-neural-core-verify-ldvgn3"  # most current; set None for default branch
REPO_DIR = Path("/content/nueronce") if Path("/content").exists() else Path.cwd()

if not (REPO_DIR / "pyproject.toml").exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(REPO_DIR)]
    subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Pin datasets below v4 because codeparrot/github-code uses a dataset loading script.
packages = [
    "datasets>=2.19,<4",
    "huggingface_hub>=0.24",
    "tqdm>=4.66",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-build-isolation"],
    check=True,
)

print("Ready.")
print("Working directory:", Path.cwd())


Ready.
Working directory: /content/nueronce


In [2]:
# 2) Verify imports, hardware, and free disk space.

import shutil
import numpy as np
import torch
import cfna

disk = shutil.disk_usage(Path.cwd())

print("cfna:", cfna.__version__)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("gpu memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

print("free disk GB:", round(disk.free / 1e9, 2))

if disk.free < 6_000_000_000:
    print("WARNING: less than 6 GB free. Reduce corpus budgets before building.")


cfna: 0.3.0
numpy: 2.0.2
torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4
gpu memory GB: 15.64
free disk GB: 202.8


In [3]:
# 3) LARGE-CORPUS AND TRAINING CONFIGURATION

from pathlib import Path

CORPUS_DIR = Path("corpus_large")
REBUILD_CORPUS = False          # True deletes and rebuilds corpus_large.
SHARD_BYTES = 8_000_000         # More shards preserve the intended source mixture.
VAL_FRACTION = 0.02             # Whole-document/repository holdout.

# Strict/default sources.
COSMOPEDIA_BYTES = 300_000_000
PERMISSIVE_CODE_BYTES = 450_000_000
FORGELOOP_TRACE_BYTES = 100_000_000
PUBLIC_DOMAIN_MAX_BYTES = 80_000_000

# Optional source. CDLA-Sharing is commercially usable subject to its obligations,
# but it is not the same as a permissive MIT/Apache/CC0 grant.
ACCEPT_CDLASHARING = False
TINYSTORIES_BYTES = 100_000_000 if ACCEPT_CDLASHARING else 0

CODE_LANGUAGES = ["Python", "Rust", "TypeScript", "Shell"]
ALLOWED_CODE_LICENSES = {
    "mit",
    "apache-2.0",
    "bsd-2-clause",
    "bsd-3-clause",
    "isc",
    "cc0-1.0",
    "unlicense",
}

MAX_CODE_FILE_BYTES = 60_000
MIN_DOCUMENT_CHARS = 200
MAX_FORGELOOP_SFT_SEEDS = 25_000

# Base training: use the old checkpoint as initialization, but save the expanded
# run separately so the original experiment is preserved.
INITIAL_BASE_CKPT = Path("checkpoints/cfna_chat.pt")
LARGE_BASE_CKPT = Path("checkpoints/cfna_base_large.pt")
BASE_TRAIN_MINUTES = 270.0  # ~4.5 h; leave margin inside the session
BASE_SEQ_LEN = 192
BASE_BATCH_SIZE = 16
BASE_LEARNING_RATE = 5e-4

# Instruction tuning.
SFT_DIR = Path("data/cfna_forgeloop_sft")
SFT_CKPT = Path("checkpoints/cfna_forgeloop_sft.pt")
SFT_TRAIN_MINUTES = 60.0
SFT_BATCH_SIZE = 4
SFT_MAX_LEN = 1024  # 384 + long system prompt filtered ~20K pairs down to 429
SFT_LEARNING_RATE = 2e-4

SYSTEM_PROMPT = (
    "You are CFNA, a bounded software-engineering assistant. "
    "Respect authority and provenance constraints. For coding work use ForgeLoop: "
    "CONTRACT, MAP, PLAN, ACT, OBSERVE, CRITIQUE, REVISE, VERIFY, LEDGER, MEMORY. "
    "Do not claim a tool result that was not observed."
)

planned = (
    COSMOPEDIA_BYTES
    + PERMISSIVE_CODE_BYTES
    + FORGELOOP_TRACE_BYTES
    + PUBLIC_DOMAIN_MAX_BYTES
    + TINYSTORIES_BYTES
)

print("Planned external/generated corpus cap:", round(planned / 1e9, 2), "GB")
print("Corpus directory:", CORPUS_DIR)
print("Initial checkpoint:", INITIAL_BASE_CKPT)
print("Expanded base checkpoint:", LARGE_BASE_CKPT)
print("SFT checkpoint:", SFT_CKPT)


# --- Paid-tier GPU auto-scaling (T4 16GB / L4 24GB / A100 40-80GB) ---
# Bigger GPUs take bigger batches at the same LR schedule; throughput scales
# nearly linearly. Colab Pro+: use Runtime > "Run cell after disconnect"
# semantics via background execution — this notebook is resume-safe either way.
try:
    import torch
    if torch.cuda.is_available():
        _gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        _name = torch.cuda.get_device_name(0)
        if _gb >= 70:      BASE_BATCH_SIZE, BASE_SEQ_LEN = 96, 256
        elif _gb >= 38:    BASE_BATCH_SIZE, BASE_SEQ_LEN = 64, 256
        elif _gb >= 22:    BASE_BATCH_SIZE, BASE_SEQ_LEN = 32, 192
        else:              BASE_BATCH_SIZE, BASE_SEQ_LEN = 16, 192
        print(f"GPU: {_name} ({_gb:.0f} GB) -> batch {BASE_BATCH_SIZE}, seq {BASE_SEQ_LEN}")
except Exception as e:
    print("GPU probe failed, keeping defaults:", e)


# --- FAST_SESSION: fit corpus build + base train + SFT + test in ~2 hours ---
FAST_SESSION = False  # night session: full ~0.93 GB corpus budgets
if FAST_SESSION:
    COSMOPEDIA_BYTES = 120_000_000
    PERMISSIVE_CODE_BYTES = 150_000_000
    FORGELOOP_TRACE_BYTES = 40_000_000
    BASE_TRAIN_MINUTES = 55.0
    SFT_TRAIN_MINUTES = 20.0
    print("FAST_SESSION: shrunk stream budgets; base 55 min, SFT 20 min")


Planned external/generated corpus cap: 0.93 GB
Corpus directory: corpus_large
Initial checkpoint: checkpoints/cfna_chat.pt
Expanded base checkpoint: checkpoints/cfna_base_large.pt
SFT checkpoint: checkpoints/cfna_forgeloop_sft.pt
GPU: Tesla T4 (16 GB) -> batch 16, seq 192


In [4]:
# 3b) Restore yesterday's checkpoints from Drive (fresh runtimes start empty).
# Brings back the 11M base (bpb 1.785) so the SFT cells can warm-start from it
# without retraining, plus any prior 35M progress for --resume.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")
src = Path("/content/drive/MyDrive/CFNA_checkpoints")
Path("checkpoints").mkdir(exist_ok=True)
for name in ["cfna_base_large.pt", "cfna_base_large_best.pt",
             "cfna_base_large.pt.json", "cfna_forgeloop_sft.pt",
             "cfna_forgeloop_sft_best.pt", "cfna_base_35m.pt"]:
    f = src / name
    if f.exists():
        shutil.copy2(f, Path("checkpoints") / name)
        print("restored:", name)


Mounted at /content/drive
restored: cfna_base_large.pt
restored: cfna_base_large_best.pt
restored: cfna_base_large.pt.json
restored: cfna_forgeloop_sft.pt
restored: cfna_forgeloop_sft_best.pt


## 4) Build the balanced corpus

The builder below does four important things the earlier one-file manifest did not:

- It allocates a separate byte budget to each source instead of allowing the first source to consume the entire target.
- It writes multiple shards, so document-uniform sampling approximately preserves the requested mixture.
- It assigns complete documents to train or validation using a deterministic hash.
- Coding files are held out by repository name, preventing files from the same repository from leaking across splits.

Code samples are limited to permissive licenses and retain repository, path, language, and license headers. A conservative secret-pattern filter rejects suspicious files.


In [5]:
# 4) Build corpus_large and seed the ForgeLoop SFT set.

import ast
import warnings
# The permissive-code grader ast-parses downloaded third-party files; their
# sloppy string literals raise SyntaxWarning noise that says nothing about
# this pipeline. Silence that category only.
warnings.filterwarnings("ignore", category=SyntaxWarning)
import hashlib
import json
import random
import re
import shutil
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm

END_DOCUMENT = "\n\n<|END_DOCUMENT|>\n\n"
TODAY = date.today().isoformat()

if REBUILD_CORPUS and CORPUS_DIR.exists():
    shutil.rmtree(CORPUS_DIR)

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
FORGE_SEED_PATH = CORPUS_DIR / "forgeloop_sft_seed.jsonl"

SKIP_DIRS = {
    ".git", ".pytest_cache", "__pycache__", ".mypy_cache", ".ruff_cache",
    "checkpoints", "corpus", "corpus_stack", "corpus_large", "data",
    "dist", "build", "node_modules", ".venv", "venv",
}

SECRET_PATTERNS = [
    re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----"),
    re.compile(r"\bAKIA[0-9A-Z]{16}\b"),
    re.compile(r"(?i)\b(?:api[_-]?key|secret[_-]?key|access[_-]?token)\s*=\s*['\"][^'\"]{12,}"),
]

def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = "\n".join(line.rstrip() for line in text.splitlines())
    text = re.sub(r"\n{4,}", "\n\n\n", text).strip()
    return text

def has_secret_like_content(text: str) -> bool:
    return any(pattern.search(text) for pattern in SECRET_PATTERNS)

def deterministic_split(key: str, val_fraction: float = VAL_FRACTION) -> str:
    digest = hashlib.sha256(key.encode("utf-8", errors="ignore")).digest()
    value = int.from_bytes(digest[:8], "big") / 2**64
    return "val" if value < val_fraction else "train"

def safe_name(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9_.-]+", "_", value).strip("_")
    return value[:100] or "source"

class ShardedCorpusWriter:
    def __init__(self, root: Path, shard_bytes: int):
        self.root = root
        self.shard_bytes = shard_bytes
        self.text_root = root / "stack_text"
        self.text_root.mkdir(parents=True, exist_ok=True)
        self.active = {}
        self.records = []
        self.source_bytes = Counter()
        self.source_docs = Counter()

    def _key(self, meta: dict, split: str):
        return (meta["source_id"], meta["license_id"], split)

    def _open(self, meta: dict, split: str):
        key = self._key(meta, split)
        index = sum(
            1 for record in self.records
            if record["source_id"] == meta["source_id"]
            and record["license_id"] == meta["license_id"]
            and record["split"] == split
        )
        folder = self.text_root / safe_name(meta["source_id"]) / safe_name(meta["license_id"])
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / f"{split}_{index:05d}.txt"
        handle = path.open("w", encoding="utf-8")
        self.active[key] = {
            "handle": handle,
            "path": path,
            "bytes": 0,
            "docs": 0,
            "meta": dict(meta),
            "split": split,
        }
        return self.active[key]

    def _close_key(self, key):
        state = self.active.pop(key, None)
        if state is None:
            return
        state["handle"].close()
        if state["docs"] == 0:
            state["path"].unlink(missing_ok=True)
            return

        blob = state["path"].read_bytes()
        digest = hashlib.sha256(blob).hexdigest()
        meta = state["meta"]
        relative_path = state["path"].relative_to(self.root).as_posix()
        shard_id = state["path"].stem

        self.records.append({
            "document_id": f"{meta['source_id']}__{safe_name(meta['license_id'])}__{shard_id}",
            "title": f"{meta['name']} {state['split']} shard {shard_id}",
            "author": meta.get("author", "dataset"),
            "document_type": meta["role"],
            "source_id": meta["source_id"],
            "source_collection": meta["name"],
            "source_locator": meta["source_locator"],
            "files_page": meta.get("files_page", meta["source_locator"]),
            "license": meta["license"],
            "license_id": meta["license_id"],
            "commercial_use": meta.get("commercial_use", True),
            "attribution_required": meta.get("attribution_required", True),
            "language": meta.get("language", "en"),
            "publication_year": meta.get("publication_year"),
            "retrieved_at": TODAY,
            "content_hash": f"sha256:{digest}",
            "quality_score": 1.0,
            "n_bytes": len(blob),
            "n_docs": state["docs"],
            "split": state["split"],
            "bucket": meta.get("bucket", "safe_commercial"),
            "phase": meta.get("phase", 1),
            "role": meta["role"],
            "path": relative_path,
        })

    def add(self, text: str, meta: dict, split_key: str) -> int:
        text = clean_text(text)
        if len(text) < MIN_DOCUMENT_CHARS:
            return 0

        encoded = (text + END_DOCUMENT).encode("utf-8")
        split = deterministic_split(split_key)
        key = self._key(meta, split)
        state = self.active.get(key) or self._open(meta, split)

        if state["docs"] and state["bytes"] + len(encoded) > self.shard_bytes:
            self._close_key(key)
            state = self._open(meta, split)

        state["handle"].write(text)
        state["handle"].write(END_DOCUMENT)
        state["bytes"] += len(encoded)
        state["docs"] += 1

        self.source_bytes[meta["source_id"]] += len(encoded)
        self.source_docs[meta["source_id"]] += 1
        return len(encoded)

    def close(self):
        for key in list(self.active):
            self._close_key(key)

    def write_manifest(self):
        self.close()
        manifest = self.root / "manifest.jsonl"
        with manifest.open("w", encoding="utf-8") as handle:
            for record in self.records:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        return manifest

writer = ShardedCorpusWriter(CORPUS_DIR, SHARD_BYTES)

# ---------------------------------------------------------------------------
# A0) OWNER BOOKS: every .txt book committed to this repository (novels,
# psych, sociology, philosophy, grammar, science). These are the owner's
# curated public-domain texts and get their own corpus bucket. Whole-book
# split keys keep train/val separation at book level (no leakage).
# ---------------------------------------------------------------------------

BOOK_SKIP_NAMES = {"requirements.txt", "checksums.txt"}
BOOK_SKIP_DIRS = SKIP_DIRS | {"docs", "benchmarks", "metrics", "tests", "scripts", "cfna", "notebooks"}
owner_book_bytes = 0
owner_book_files = 0
for path in sorted(Path.cwd().rglob("*.txt")):
    if any(part in BOOK_SKIP_DIRS or part.endswith(".egg-info") for part in path.parts):
        continue
    if path.name.lower() in BOOK_SKIP_NAMES:
        continue
    try:
        text = clean_text(path.read_text(encoding="utf-8", errors="replace"))
    except OSError:
        continue
    if len(text) < MIN_DOCUMENT_CHARS:
        continue
    relative = path.relative_to(Path.cwd()).as_posix()
    book_meta = {
        "source_id": "owner_books",
        "name": f"Owner book: {path.stem}",
        "role": "base_pretraining_books",
        "source_locator": f"{REPO_URL} :: {relative}",
        "files_page": REPO_URL,
        "license": "Public domain (owner-curated classic texts)",
        "license_id": "public-domain",
        "commercial_use": True,
        "attribution_required": False,
        "bucket": "owner_books",
        "phase": 1,
        "publication_year": None,
        "retrieved_at": TODAY,
    }
    # chunk big books into ~96KB paragraph-aligned docs; split key stays the
    # whole book so all chunks land in the same train/val side
    chunk, size = [], 0
    for para in text.split("\n\n"):
        chunk.append(para)
        size += len(para)
        if size >= 96_000:
            writer.add("\n\n".join(chunk), book_meta, split_key=f"book::{relative}")
            owner_book_bytes += size
            chunk, size = [], 0
    if chunk:
        writer.add("\n\n".join(chunk), book_meta, split_key=f"book::{relative}")
        owner_book_bytes += size
    owner_book_files += 1
print(f"owner books: {owner_book_files} files / {owner_book_bytes/1e6:.1f} MB -> bucket owner_books")

# ---------------------------------------------------------------------------
# A) This repository: owned/controlled Apache-2.0 code and documentation.
# ---------------------------------------------------------------------------

repo_meta = {
    "source_id": "nueronce_repository",
    "name": "Nueronce / CFNA repository",
    "role": "base_pretraining_code",
    "source_locator": REPO_URL,
    "files_page": REPO_URL,
    "license": "Apache-2.0",
    "license_id": "apache-2.0",
    "commercial_use": True,
    "attribution_required": True,
    "bucket": "safe_commercial",
    "phase": 1,
}

allowed_suffixes = {
    ".py", ".md", ".rst", ".toml", ".json", ".yaml", ".yml", ".sh", ".rs", ".ts", ".tsx"
}

for path in sorted(Path.cwd().rglob("*")):
    if not path.is_file() or path.suffix.lower() not in allowed_suffixes:
        continue
    if any(part in SKIP_DIRS for part in path.parts):
        continue
    try:
        text = path.read_text(encoding="utf-8")
    except (UnicodeDecodeError, OSError):
        continue
    if has_secret_like_content(text):
        continue
    relative = path.relative_to(Path.cwd()).as_posix()
    document = f"Repository: {REPO_URL}\nPath: {relative}\nLanguage: Apache-2.0\n\n{text}"
    writer.add(document, repo_meta, split_key=relative)

print("Local repository bytes:", f"{writer.source_bytes['nueronce_repository']/1e6:.2f} MB")

# ---------------------------------------------------------------------------
# B) Public-domain books and U.S.-government sources already registered in CFNA.
# ---------------------------------------------------------------------------

try:
    from cfna.corpus.build import build_corpus

    temporary_pd = CORPUS_DIR / "_public_domain_build"
    pd_records = build_corpus(temporary_pd)

    pd_used = 0
    for record in pd_records:
        if pd_used >= PUBLIC_DOMAIN_MAX_BYTES:
            break
        source_path = temporary_pd / record.path
        text = source_path.read_text(encoding="utf-8", errors="replace")
        meta = {
            "source_id": f"public_domain_{record.source_collection}",
            "name": record.source_collection,
            "role": "base_pretraining_books",
            "source_locator": record.source_locator,
            "files_page": record.source_locator,
            "license": record.license,
            "license_id": record.license_id,
            "commercial_use": record.commercial_use,
            "attribution_required": record.attribution_required,
            "bucket": "public_domain",
            "phase": 1,
            "author": record.author,
            "publication_year": record.publication_year,
        }
        pd_used += writer.add(text, meta, split_key=record.document_id)

    print("Public-domain bytes:", f"{pd_used/1e6:.2f} MB")
except Exception as exc:
    print("WARNING: public-domain acquisition skipped:", repr(exc))

# ---------------------------------------------------------------------------
# C) Balanced language sources.
# ---------------------------------------------------------------------------

def stream_text_source(
    dataset_name: str,
    budget_bytes: int,
    meta: dict,
    text_fields=("text",),
    dataset_config=None,
):
    if budget_bytes <= 0:
        return 0

    kwargs = {
        "path": dataset_name,
        "split": "train",
        "streaming": True,
    }
    if dataset_config:
        kwargs["name"] = dataset_config

    dataset = load_dataset(**kwargs)
    used = 0
    accepted = 0

    for index, row in enumerate(tqdm(dataset, desc=meta["source_id"])):
        text = None
        for field in text_fields:
            value = row.get(field)
            if isinstance(value, str) and value.strip():
                text = value
                break

        if not text:
            continue

        text = clean_text(text)
        if len(text) < MIN_DOCUMENT_CHARS:
            continue

        size = len((text + END_DOCUMENT).encode("utf-8"))
        if used + size > budget_bytes:
            break

        split_key = f"{meta['source_id']}:{index}:{hashlib.sha256(text[:500].encode()).hexdigest()}"
        used += writer.add(text, meta, split_key)
        accepted += 1

    print(meta["source_id"], "accepted docs:", accepted, "bytes:", round(used / 1e6, 2), "MB")
    return used

cosmopedia_meta = {
    "source_id": "cosmopedia_100k",
    "name": "Cosmopedia-100k",
    "role": "base_pretraining",
    "source_locator": "https://huggingface.co/datasets/HuggingFaceTB/cosmopedia-100k",
    "files_page": "https://huggingface.co/datasets/HuggingFaceTB/cosmopedia-100k/tree/main",
    "license": "Apache-2.0",
    "license_id": "apache-2.0",
    "commercial_use": True,
    "attribution_required": True,
    "bucket": "safe_commercial",
    "phase": 1,
}

stream_text_source(
    "HuggingFaceTB/cosmopedia-100k",
    COSMOPEDIA_BYTES,
    cosmopedia_meta,
    text_fields=("text", "markdown", "content"),
)

if TINYSTORIES_BYTES:
    tinystories_meta = {
        "source_id": "tinystories",
        "name": "TinyStories",
        "role": "base_pretraining",
        "source_locator": "https://huggingface.co/datasets/roneneldan/TinyStories",
        "files_page": "https://huggingface.co/datasets/roneneldan/TinyStories/tree/main",
        "license": "CDLA-Sharing-1.0",
        "license_id": "cdla-sharing-1.0",
        "commercial_use": True,
        "attribution_required": True,
        "bucket": "share_alike_review",
        "phase": 1,
    }
    stream_text_source(
        "roneneldan/TinyStories",
        TINYSTORIES_BYTES,
        tinystories_meta,
    )

# ---------------------------------------------------------------------------
# D) Permissive code + original CFNA ForgeLoop traces.
# ---------------------------------------------------------------------------

def python_facts(code_text: str) -> dict:
    try:
        tree = ast.parse(code_text)
    except SyntaxError:
        return {}

    functions = [node for node in ast.walk(tree) if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
    classes = [node for node in ast.walk(tree) if isinstance(node, ast.ClassDef)]
    imports = []
    calls = []

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            imports.extend(alias.name for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            imports.append(node.module or "")
        elif isinstance(node, ast.Call):
            fn = node.func
            if isinstance(fn, ast.Name):
                calls.append(fn.id)
            elif isinstance(fn, ast.Attribute):
                calls.append(fn.attr)

    primary = functions[0] if functions else (classes[0] if classes else None)
    signature = None
    symbol = None

    if isinstance(primary, (ast.FunctionDef, ast.AsyncFunctionDef)):
        symbol = primary.name
        args = [arg.arg for arg in primary.args.args]
        signature = f"{primary.name}({', '.join(args)})"
    elif isinstance(primary, ast.ClassDef):
        symbol = primary.name
        signature = f"class {primary.name}"

    return {
        "symbol": symbol,
        "signature": signature,
        "functions": [node.name for node in functions[:8]],
        "classes": [node.name for node in classes[:8]],
        "imports": sorted({item for item in imports if item})[:8],
        "calls": sorted(set(calls))[:10],
        "branches": sum(isinstance(node, (ast.If, ast.Match, ast.Try)) for node in ast.walk(tree)),
        "loops": sum(isinstance(node, (ast.For, ast.AsyncFor, ast.While)) for node in ast.walk(tree)),
        "raises": sum(isinstance(node, ast.Raise) for node in ast.walk(tree)),
    }

def make_forgeloop_example(repo: str, path: str, language: str, license_id: str, code_text: str):
    lines = code_text.count("\n") + 1
    facts = python_facts(code_text) if language == "Python" else {}
    symbol = facts.get("symbol") or Path(path).name
    signature = facts.get("signature") or f"{language} file {Path(path).name}"
    imports = ", ".join(facts.get("imports", [])) or "not statically summarized"
    calls = ", ".join(facts.get("calls", [])) or "not statically summarized"

    prompt = (
        f"Review {signature} in {path}. Build a bounded test-first implementation plan. "
        f"Repository={repo}; license={license_id}."
    )

    response = (
        "CONTRACT: Preserve the public API, edit only the requested scope, do not access "
        "secrets or the network, and treat tests/tool output as the authority.\n"
        f"MAP: language={language}; symbol={symbol}; lines={lines}; imports={imports}; calls={calls}; "
        f"branches={facts.get('branches', 'unknown')}; loops={facts.get('loops', 'unknown')}; "
        f"explicit_raises={facts.get('raises', 'unknown')}.\n"
        "PLAN: Read the implementation and nearby tests; identify invariants and failure paths; "
        "write or select one failing test; make the smallest compatible patch.\n"
        "ACT: Apply one bounded change. Do not rewrite unrelated modules.\n"
        "OBSERVE: Record the exact test, type-check, lint, or runtime output. Never invent success.\n"
        "CRITIQUE: Compare the observation with the contract and check regressions, edge cases, "
        "error handling, and backward compatibility.\n"
        "REVISE: If verification fails, update the hypothesis and make one additional bounded change.\n"
        "VERIFY: Run the targeted test first, then the relevant suite. Stop on unexplained failure.\n"
        f"LEDGER: Retain repo={repo}, path={path}, license={license_id}, tool commands, and result hashes.\n"
        "MEMORY: Store only the verified invariant and reusable test lesson; discard unverified guesses."
    )

    document = (
        "<|CFNA_FORGELOOP|>\n"
        f"User: {prompt}\n"
        f"Assistant: {response}\n"
        "<|END_CFNA_FORGELOOP|>"
    )
    return document, {"prompt": prompt, "response": response, "kind": "forgeloop", "repo": repo, "path": path}

code_used = 0
trace_used = 0
sft_seed_count = 0

# The loader supports filtering by languages/licenses. A fallback manually filters
# rows if a datasets/codeparrot version ignores those arguments.
try:
    code_dataset = load_dataset(
        "codeparrot/github-code",
        split="train",
        streaming=True,
        languages=CODE_LANGUAGES,
        licenses=sorted(ALLOWED_CODE_LICENSES),
    )
except Exception as filtered_exc:
    print("Filtered CodeParrot loader failed; trying manual filtering:", repr(filtered_exc))
    code_dataset = load_dataset(
        "codeparrot/github-code",
        split="train",
        streaming=True,
    )

FORGE_SEED_PATH.parent.mkdir(parents=True, exist_ok=True)
with FORGE_SEED_PATH.open("w", encoding="utf-8") as seed_handle:
    for row in tqdm(code_dataset, desc="permissive_code"):
        if code_used >= PERMISSIVE_CODE_BYTES and trace_used >= FORGELOOP_TRACE_BYTES:
            break

        language = str(row.get("language") or "")
        license_id = str(row.get("license") or "").lower()
        code_text = str(row.get("code") or "")
        repo = str(row.get("repo_name") or "unknown_repo")
        path = str(row.get("path") or "unknown_path")

        if language not in CODE_LANGUAGES:
            continue
        if license_id not in ALLOWED_CODE_LICENSES:
            continue
        if not (MIN_DOCUMENT_CHARS <= len(code_text) <= MAX_CODE_FILE_BYTES):
            continue
        if has_secret_like_content(code_text):
            continue
        if any(token in path.lower() for token in ("vendor/", "node_modules/", "min.js", ".lock")):
            continue

        source_id = f"codeparrot_{language.lower()}_{license_id}"
        meta = {
            "source_id": source_id,
            "name": f"CodeParrot permissive {language} ({license_id})",
            "role": "base_pretraining_code",
            "source_locator": "https://huggingface.co/datasets/codeparrot/github-code",
            "files_page": "https://huggingface.co/datasets/codeparrot/github-code/tree/main",
            "license": license_id,
            "license_id": license_id,
            "commercial_use": True,
            "attribution_required": license_id not in {"cc0-1.0", "unlicense"},
            "bucket": "safe_commercial",
            "phase": 3,
        }

        split_key = f"repository:{repo}"
        if code_used < PERMISSIVE_CODE_BYTES:
            code_document = (
                f"Repository: {repo}\nPath: {path}\nLanguage: {language}\n"
                f"License: {license_id}\n\n{code_text}"
            )
            code_used += writer.add(code_document, meta, split_key=split_key)

        if trace_used < FORGELOOP_TRACE_BYTES:
            trace_document, sft_example = make_forgeloop_example(
                repo, path, language, license_id, code_text
            )
            trace_meta = dict(meta)
            trace_meta.update({
                "source_id": f"forgeloop_{language.lower()}_{license_id}",
                "name": f"CFNA ForgeLoop traces from permissive {language}",
                "role": "agentic_coding_scaffold",
                "phase": 4,
            })
            trace_used += writer.add(trace_document, trace_meta, split_key=split_key)

            if sft_seed_count < MAX_FORGELOOP_SFT_SEEDS:
                sft_example["split_key"] = split_key
                sft_example["license"] = license_id
                seed_handle.write(json.dumps(sft_example, ensure_ascii=False) + "\n")
                sft_seed_count += 1

manifest_path = writer.write_manifest()

print("\nCorpus build complete.")
print("Manifest:", manifest_path)
print("ForgeLoop SFT seeds:", sft_seed_count)
print("Code bytes:", round(code_used / 1e6, 2), "MB")
print("ForgeLoop trace bytes:", round(trace_used / 1e6, 2), "MB")
print("Total manifest bytes:", round(sum(r["n_bytes"] for r in writer.records) / 1e9, 3), "GB")
print("\nBytes by source:")
for source_id, size in writer.source_bytes.most_common():
    print(f"  {source_id:45s} {size/1e6:10.2f} MB  docs={writer.source_docs[source_id]:,}")

owner books: 21 files / 23.5 MB -> bucket owner_books
Local repository bytes: 2.08 MB
Public-domain bytes: 31.04 MB


README.md:   0%|          | 0.00/944 [00:00<?, ?B/s]

cosmopedia_100k: 0it [00:00, ?it/s]

cosmopedia_100k accepted docs: 79621 bytes: 300.0 MB


README.md:   0%|          | 0.00/7.54k [00:00<?, ?B/s]

github-code.py:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

The repository for codeparrot/github-code contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/codeparrot/github-code.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


permissive_code: 0it [00:00, ?it/s]


Corpus build complete.
Manifest: corpus_large/manifest.jsonl
ForgeLoop SFT seeds: 25000
Code bytes: 450.0 MB
ForgeLoop trace bytes: 100.0 MB
Total manifest bytes: 0.907 GB

Bytes by source:
  cosmopedia_100k                                   300.00 MB  docs=79,621
  codeparrot_python_apache-2.0                      129.56 MB  docs=17,369
  codeparrot_python_mit                             125.15 MB  docs=25,326
  codeparrot_python_bsd-3-clause                     69.60 MB  docs=9,710
  codeparrot_typescript_mit                          51.09 MB  docs=13,208
  forgeloop_python_mit                               29.40 MB  docs=19,929
  owner_books                                        23.78 MB  docs=253
  forgeloop_python_apache-2.0                        21.31 MB  docs=13,807
  public_domain_project_gutenberg (canonical cache)      16.43 MB  docs=25
  forgeloop_typescript_mit                           15.23 MB  docs=10,408
  codeparrot_typescript_apache-2.0                   14.21 MB  

In [6]:
# 5) Validate provenance, paths, licenses, and real train/validation separation.

import hashlib
import json
from collections import Counter, defaultdict
from pathlib import Path

manifest_path = CORPUS_DIR / "manifest.jsonl"
records = [
    json.loads(line)
    for line in manifest_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert records, "Corpus manifest is empty."

splits = Counter()
roles = Counter()
licenses = Counter()
source_bytes = Counter()
hash_to_split = defaultdict(set)

for record in records:
    path = CORPUS_DIR / record["path"]
    assert path.is_file(), f"Missing corpus shard: {path}"
    assert record["split"] in {"train", "val"}, record
    assert record["source_locator"].startswith("https://"), record["document_id"]
    assert record["license_id"], record["document_id"]

    actual_hash = "sha256:" + hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual_hash == record["content_hash"], record["document_id"]

    splits[record["split"]] += 1
    roles[record["role"]] += record.get("n_bytes", 0)
    licenses[record["license_id"]] += record.get("n_bytes", 0)
    source_bytes[record["source_id"]] += record.get("n_bytes", 0)
    hash_to_split[record["content_hash"]].add(record["split"])

assert splits["train"] > 0, "No training shards."
assert splits["val"] > 0, "No validation shards."
assert not any(len(value) > 1 for value in hash_to_split.values()), (
    "Identical shard bytes appear in both train and validation."
)

print("Manifest shards:", len(records))
print("Splits:", dict(splits))
print("Total bytes:", f"{sum(r['n_bytes'] for r in records)/1e9:.3f} GB")
print("\nRole mixture:")
for role, size in roles.most_common():
    print(f"  {role:32s} {size/1e6:10.2f} MB")

print("\nLicense mixture:")
for license_id, size in licenses.most_common():
    print(f"  {license_id:32s} {size/1e6:10.2f} MB")

print("\nValidation passed.")


Manifest shards: 210
Splits: {'train': 159, 'val': 51}
Total bytes: 0.907 GB

Role mixture:
  base_pretraining_code                452.09 MB
  base_pretraining                     300.00 MB
  agentic_coding_scaffold              100.00 MB
  base_pretraining_books                54.82 MB

License mixture:
  apache-2.0                           494.43 MB
  mit                                  242.64 MB
  bsd-3-clause                          89.00 MB
  public-domain-us                      28.14 MB
  public-domain                         23.78 MB
  bsd-2-clause                          16.59 MB
  unlicense                              5.03 MB
  public-domain-usgov                    2.90 MB
  isc                                    2.65 MB
  cc0-1.0                                1.74 MB

Validation passed.


## 6) Write the GPU base trainer

This trainer keeps the repository's 11.1M-parameter configuration, adds CUDA/AMP support, supports initialization from the earlier TinyStories checkpoint, and then resumes from the expanded checkpoint on later runs.

The output checkpoint is separate from `cfna_chat.pt`.


In [7]:
# 6) Create a resumable CUDA/AMP base-training script.

import textwrap
from pathlib import Path

trainer_path = Path("scripts/train_checkpoint_gpu_large.py")

trainer_code = r"""
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import math
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch

from cfna.corpus.dataset import ByteCorpus, val_batches
from cfna.model import CFNAModel, ModelConfig

LN2 = math.log(2.0)

def chat_config() -> ModelConfig:
    return ModelConfig(
        byte_embed_dim=64,
        d_local=128,
        d_model=256,
        p_max=48,
        physical_blocks=3,
        logical_depth=4,
        n_heads=8,
        unit_window=48,
        decoder_window=64,
        decoder_layers=3,
        d_state=16,
        channel_dim=24,
        ret_byte_dim=32,
        min_patch=3,
        max_patch=24,
        boundary_loss_weight=0.2,
    )

def amp_context(device, enabled):
    if enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

@torch.no_grad()
def heldout_bpb(model, batches, device, amp_enabled):
    if not batches:
        return float("nan")
    model.eval()
    values = []
    for batch in batches:
        with amp_context(device, amp_enabled):
            values.append(model.lm_loss(batch).float().item())
    return float(np.mean(values) / LN2)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--init", default="")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--minutes", type=float, default=120.0)
    ap.add_argument("--seq", type=int, default=192)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--log-every", type=int, default=25)
    ap.add_argument("--val-batches", type=int, default=6)
    ap.add_argument("--no-amp", action="store_true")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp_enabled = device.type == "cuda" and not args.no_amp

    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    print("device:", device)
    if device.type == "cuda":
        print("gpu:", torch.cuda.get_device_name(0))
    print("amp:", amp_enabled)

    train = ByteCorpus(args.corpus, "train")
    val = ByteCorpus(args.corpus, "val")
    valb = val_batches(
        val,
        args.seq,
        args.batch,
        max_batches=args.val_batches,
        device=device,
    )

    print(
        f"train {train.total_bytes/1e6:.2f} MB / {len(train.docs)} shards | "
        f"held-out {val.total_bytes/1e6:.2f} MB / {len(val.docs)} shards"
    )

    cfg = chat_config()
    model = CFNAModel(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    best_path = Path(str(out).replace(".pt", "") + "_best.pt")

    def _atomic_torch_save(payload, dest):
        tmp = Path(str(dest) + ".tmp")
        torch.save(payload, tmp)
        tmp.replace(dest)

    history = []
    step = 0
    loaded_from = None

    if args.resume and out.exists():
        loaded_from = out
        checkpoint = torch.load(out, map_location=device, weights_only=False)
        if checkpoint.get("config") != vars(cfg):
            raise ValueError("Expanded checkpoint config does not match trainer config.")
        model.load_state_dict(checkpoint["state_dict"])
        if "optimizer" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer"])
            # load_state_dict restores the OLD lr; re-apply the CLI lr so the
            # convergence ladder (resume with a lower --lr) actually works.
            for group in optimizer.param_groups:
                group["lr"] = args.lr
        step = int(checkpoint.get("step", 0))
        history = list(checkpoint.get("history", []))
    elif args.init and Path(args.init).exists():
        loaded_from = Path(args.init)
        checkpoint = torch.load(loaded_from, map_location=device, weights_only=False)
        if checkpoint.get("config") != vars(cfg):
            raise ValueError("Initialization checkpoint config does not match trainer config.")
        model.load_state_dict(checkpoint["state_dict"])
        step = int(checkpoint.get("step", 0))
        history = list(checkpoint.get("history", []))
        # New corpus and LR: deliberately reset optimizer state.

    if loaded_from:
        print(f"loaded weights from {loaded_from} at global step {step}")
    else:
        print("starting from random initialization")

    print(
        f"model: {model.num_params():,} params | seq {args.seq} | batch {args.batch} | "
        f"budget {args.minutes:.1f} min"
    )

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    rng = np.random.default_rng(args.seed + step)
    start = time.time()
    run_steps = 0
    best_val = min(
        (item["heldout_bpb"] for item in history if "heldout_bpb" in item),
        default=float("inf"),
    )

    def save():
        _atomic_torch_save(
            {
                "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "optimizer": optimizer.state_dict(),
                "config": vars(cfg),
                "step": step,
                "history": history,
                "corpus": str(args.corpus),
            },
            out,
        )
        Path(str(out) + ".json").write_text(
            json.dumps(
                {
                    "config": vars(cfg),
                    "step": step,
                    "history": history,
                    "corpus": str(args.corpus),
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    while (time.time() - start) < args.minutes * 60:
        array = train.sample_batch(args.seq, args.batch, rng)
        batch = torch.from_numpy(array).to(device, non_blocking=True)

        model.train()
        optimizer.zero_grad(set_to_none=True)

        with amp_context(device, amp_enabled):
            loss, parts = model.loss(batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        step += 1
        run_steps += 1

        if run_steps % args.log_every == 0:
            val_score = heldout_bpb(model, valb, device, amp_enabled)
            elapsed = time.time() - start
            bytes_seen = run_steps * args.batch * max(1, args.seq - 1)
            throughput = bytes_seen / max(elapsed, 1e-9)
            record = {
                "step": step,
                "run_step": run_steps,
                "train_bpb": float(parts["bpb"]),
                "heldout_bpb": val_score,
                "minutes": elapsed / 60,
                "bytes_per_second": throughput,
            }
            history.append(record)
            if val_score < best_val:
                best_val = val_score
                _atomic_torch_save(
                    {
                        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                        "config": vars(cfg),
                        "step": step,
                        "history": history,
                        "corpus": str(args.corpus),
                        "best_heldout_bpb": best_val,
                    },
                    best_path,
                )
                print(f"  new best held-out bpb {best_val:.3f} -> {best_path}", flush=True)
            print(
                f"step {step:6d} | run {run_steps:5d} | {elapsed/60:6.1f}m | "
                f"train bpb {record['train_bpb']:.3f} | held-out bpb {val_score:.3f} | "
                f"{throughput:,.0f} byte-targets/s",
                flush=True,
            )
            save()

    if run_steps and run_steps % args.log_every:
        val_score = heldout_bpb(model, valb, device, amp_enabled)
        elapsed = time.time() - start
        history.append(
            {
                "step": step,
                "run_step": run_steps,
                "train_bpb": float(parts["bpb"]),
                "heldout_bpb": val_score,
                "minutes": elapsed / 60,
            }
        )
        best_val = min(best_val, val_score)

    save()
    print(
        f"saved checkpoint -> {out} "
        f"({run_steps} new steps; global step {step}; best held-out bpb {best_val:.3f})"
    )

if __name__ == "__main__":
    main()
"""

trainer_path.write_text(textwrap.dedent(trainer_code).lstrip(), encoding="utf-8")
print("Wrote:", trainer_path)


Wrote: scripts/train_checkpoint_gpu_large.py


In [8]:
# Auto-backup to Drive every 30 min while training runs — never lose a night again
import shutil, time, threading
from pathlib import Path
def _auto_backup():
    dst = Path("/content/drive/MyDrive/CFNA_checkpoints"); dst.mkdir(parents=True, exist_ok=True)
    while BASE_PROC.poll() is None:
        time.sleep(1800)
        for f in ["checkpoints/cfna_base_large.pt", "checkpoints/cfna_base_large_best.pt",
                  "checkpoints/cfna_base_large.pt.json"]:
            if Path(f).exists(): shutil.copy2(f, dst / Path(f).name)
        print("auto-backup ✓", time.strftime("%H:%M"))
threading.Thread(target=_auto_backup, daemon=True).start()
print("auto-backup armed: every 30 min")

auto-backup armed: every 30 min


Exception in thread Thread-7 (_auto_backup):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_1431/1873773478.py", line 6, in _auto_backup


In [9]:
# 7) Base pretraining — runs as a BACKGROUND process so the notebook stays
#    usable: monitor with cell 7a, probe inference with cell 7b, and cell 7c
#    blocks until training finishes before the SFT stages.
import os, subprocess, sys
from pathlib import Path

command = [
    sys.executable, "-u", "scripts/train_checkpoint_gpu_large.py",
    "--corpus", str(CORPUS_DIR), "--out", str(LARGE_BASE_CKPT),
    "--minutes", str(BASE_TRAIN_MINUTES), "--seq", str(BASE_SEQ_LEN),
    "--batch", str(BASE_BATCH_SIZE), "--lr", str(BASE_LEARNING_RATE),
    "--log-every", "25",
]
if LARGE_BASE_CKPT.exists():
    command.append("--resume")
elif INITIAL_BASE_CKPT.exists():
    command += ["--init", str(INITIAL_BASE_CKPT)]

env = os.environ.copy()
env["PYTHONPATH"] = str(Path.cwd()) + os.pathsep + env.get("PYTHONPATH", "")
Path("logs").mkdir(exist_ok=True)
BASE_LOG = Path("logs/base_train.log")
# start_new_session=True detaches the trainer from the notebook's process
# group, so interrupting a cell (Colab Stop button) can no longer SIGINT the
# background run. To stop training deliberately: BASE_PROC.terminate()
BASE_PROC = subprocess.Popen(command, cwd=Path.cwd(), env=env,
                             stdout=BASE_LOG.open("w"), stderr=subprocess.STDOUT,
                             text=True, start_new_session=True)
print("Launched background training, PID", BASE_PROC.pid)
print("Log:", BASE_LOG, "| latest ckpt:", LARGE_BASE_CKPT,
      "| best ckpt:", str(LARGE_BASE_CKPT).replace(".pt", "") + "_best.pt")


Launched background training, PID 12749
Log: logs/base_train.log | latest ckpt: checkpoints/cfna_base_large.pt | best ckpt: checkpoints/cfna_base_large_best.pt


In [57]:
# 7a) Monitor the background run (rerun this cell anytime)
import re
alive = BASE_PROC.poll() is None
tail = BASE_LOG.read_text(errors="replace").splitlines()[-12:] if BASE_LOG.exists() else []
print("training alive:", alive, "" if alive else f"(exit code {BASE_PROC.returncode})")
print("\n".join(tail))
bpbs = [float(m.group(1)) for l in (BASE_LOG.read_text(errors="replace").splitlines() if BASE_LOG.exists() else [])
        if (m := re.search(r"held-out bpb ([0-9.]+)", l))]
if bpbs:
    print(f"\nheld-out bpb: start {bpbs[0]:.3f} -> now {bpbs[-1]:.3f} (best {min(bpbs):.3f}; SFT gate < 1.8)")


training alive: True 
step 108798 | run 34825 |  229.4m | train bpb 1.126 | held-out bpb 1.386 | 7,732 byte-targets/s
step 108823 | run 34850 |  229.6m | train bpb 1.381 | held-out bpb 1.382 | 7,732 byte-targets/s
step 108848 | run 34875 |  229.7m | train bpb 0.815 | held-out bpb 1.384 | 7,732 byte-targets/s
step 108873 | run 34900 |  229.9m | train bpb 1.393 | held-out bpb 1.393 | 7,732 byte-targets/s
step 108898 | run 34925 |  230.1m | train bpb 1.254 | held-out bpb 1.389 | 7,732 byte-targets/s
step 108923 | run 34950 |  230.2m | train bpb 0.991 | held-out bpb 1.396 | 7,732 byte-targets/s
step 108948 | run 34975 |  230.4m | train bpb 0.835 | held-out bpb 1.393 | 7,732 byte-targets/s
step 108973 | run 35000 |  230.5m | train bpb 1.133 | held-out bpb 1.390 | 7,732 byte-targets/s
step 108998 | run 35025 |  230.7m | train bpb 1.501 | held-out bpb 1.393 | 7,733 byte-targets/s
step 109023 | run 35050 |  230.9m | train bpb 1.517 | held-out bpb 1.397 | 7,733 byte-targets/s
step 109048 | run 

In [58]:
# 7b) LIVE INFERENCE PROBE — test generation from the newest checkpoint while
#     training keeps running. Loads on CPU so it never touches the GPU mid-step;
#     checkpoint writes are atomic, so reads are always a consistent file.
import torch
from pathlib import Path
from cfna.model import CFNAModel, ModelConfig

best = Path(str(LARGE_BASE_CKPT).replace(".pt", "") + "_best.pt")
ck_path = best if best.exists() else LARGE_BASE_CKPT
if not ck_path.exists():
    print("no checkpoint yet — wait for the first save (every 25 logged steps)")
else:
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    m = CFNAModel(ModelConfig(**ck["config"]))
    m.load_state_dict(ck["state_dict"]); m.eval()
    print(f"probing {ck_path} @ step {ck.get('step')} "
          f"(best bpb {ck.get('best_heldout_bpb', 'n/a')})\n")
    for prompt in ["The nature of human understanding is",
                   "Once upon a time", "def add(a, b):"]:
        out = m.generate(prompt.encode(), max_new=90, temperature=0.0, greedy=True)
        text = out.decode("utf-8", errors="replace") if isinstance(out, (bytes, bytearray)) else str(out)
        print(f">>> {prompt!r}\n{prompt + text}\n")


probing checkpoints/cfna_base_large_best.pt @ step 73199 (best bpb 1.3023699245010754)

>>> 'The nature of human understanding is'
The nature of human understanding is a significant concern in the world of construction and international relations and politi

>>> 'Once upon a time'
Once upon a time of the state of the state of the state of the state of the people of the people.

I have 

>>> 'def add(a, b):'
def add(a, b):
                                                                                         



In [ ]:
# 7c) GATE: block here until base training finishes (run before the SFT cells)
rc = BASE_PROC.wait()
print("base training finished with exit code", rc)
assert rc == 0, "base training failed — check logs/base_train.log"


base training finished with exit code -15


AssertionError: base training failed — check logs/base_train.log

In [ ]:
# 7b) THE 35M RUNG — tonight's main event (~4.5 h, blocking; keep the tab open
# on regular Pro). NIGHT ORDER: cells 1-5 (corpus, ~25 min at full budgets) ->
# cell 9 -> 11 (SFT on the restored 11M base: ~20K pairs now, early-stops
# itself) -> 12 (chat!) -> THEN this cell for the rest of the night -> 13.
# Resume-safe: if the runtime dies, rerun; --resume continues.
# Fresh base_35m pretrain on the same corpus_large via the repo's preset-aware
# trainer (scripts/train_checkpoint.py). Do NOT warm-start from the 11M
# checkpoint: the architectures differ. Resume-safe; rerun until held-out
# bpb <= 1.5, then point the SFT cells' INITIAL checkpoint at this file.
RUN_35M = True
if RUN_35M:
    !python scripts/train_checkpoint.py --preset base_35m --corpus corpus_large \
        --minutes {BASE_TRAIN_MINUTES} --seq 192 --batch {BASE_BATCH_SIZE} \
        --lr 3e-4 --amp --resume --out checkpoints/cfna_base_35m.pt


In [ ]:
# 8) Raw base-model diagnostics.
# Test completion, not chat behavior, before instruction tuning.

import torch
from cfna.chat import load_checkpoint

base_model, base_state = load_checkpoint(str(LARGE_BASE_CKPT))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = base_model.to(device).eval()

@torch.no_grad()
def complete_bytes(model, prompt: str, max_new: int = 120, temperature: float = 0.0, max_ctx: int = 256):
    ids = list(prompt.encode("utf-8"))
    generated = bytearray()

    for _ in range(max_new):
        context = torch.tensor([ids[-max_ctx:]], dtype=torch.long, device=device)
        logits, _ = model(context)
        next_logits = logits[0, -1]

        if temperature <= 0:
            next_id = int(next_logits.argmax())
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_id = int(torch.multinomial(probs, 1))

        ids.append(next_id)
        generated.append(next_id)

    return prompt + generated.decode("utf-8", errors="replace")

for prompt in [
    "Once upon a time, a young inventor found ",
    "def validate_manifest(records):\n    ",
    "<|CFNA_FORGELOOP|>\nUser: Review a parser that fails on empty input.\nAssistant: ",
]:
    print("=" * 80)
    print(complete_bytes(base_model, prompt, max_new=100, temperature=0.0))


## 9) Build the instruction-tuning set

The SFT set mixes:

- Grounded ForgeLoop examples generated while ingesting permissive code.
- English OpenAssistant prompt/assistant pairs.
- The repository's small hand-written turn-taking set.

The split is deterministic, and the response-only mask ensures the model is trained to produce the assistant bytes rather than memorize the user's prompt.


In [ ]:
# 9) Build ForgeLoop + general-assistant SFT JSONL files.

import hashlib
import json
import random
from pathlib import Path

from datasets import load_dataset
from cfna.training.dialogue_data import SFT_DATASET, encode_example

SFT_DIR.mkdir(parents=True, exist_ok=True)

MAX_GENERAL_PAIRS = 20_000
MAX_FORGE_PAIRS = 20_000

examples = []

# A) Grounded ForgeLoop examples produced from accepted permissive code.
seed_path = CORPUS_DIR / "forgeloop_sft_seed.jsonl"
if seed_path.exists():
    forge_rows = [
        json.loads(line)
        for line in seed_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    random.Random(42).shuffle(forge_rows)
    examples.extend(forge_rows[:MAX_FORGE_PAIRS])

# B) OpenAssistant English prompt -> assistant responses.
oasst = load_dataset("OpenAssistant/oasst1", split="train")
by_id = {row["message_id"]: row for row in oasst}

general_count = 0
for row in oasst:
    if general_count >= MAX_GENERAL_PAIRS:
        break
    if row.get("role") != "assistant" or row.get("lang") != "en":
        continue

    parent = by_id.get(row.get("parent_id"))
    if not parent:
        continue
    if parent.get("role") not in {"prompter", "user"} or parent.get("lang") != "en":
        continue

    prompt = str(parent.get("text") or "").strip()
    response = str(row.get("text") or "").strip()

    if not prompt or not response:
        continue
    if len(prompt) > 700 or len(response) > 900:
        continue

    full_bytes, mask = encode_example(prompt, response, system=SYSTEM_PROMPT)
    if len(full_bytes) > SFT_MAX_LEN:
        continue
    if not any(mask):
        continue

    examples.append({
        "prompt": prompt,
        "response": response,
        "kind": "oasst1",
        "source": "OpenAssistant/oasst1",
        "license": "Apache-2.0",
    })
    general_count += 1

# C) Small built-in turn-taking set.
for prompt, response in SFT_DATASET:
    examples.append({
        "prompt": prompt,
        "response": response,
        "kind": "cfna_builtin",
        "source": "cfna.training.dialogue_data",
        "license": "Apache-2.0",
    })

# Deduplicate exact prompt/response pairs and enforce the byte limit.
deduped = []
seen = set()

for row in examples:
    key = hashlib.sha256(
        (row["prompt"] + "\0" + row["response"]).encode("utf-8")
    ).hexdigest()
    if key in seen:
        continue

    full_bytes, mask = encode_example(
        row["prompt"],
        row["response"],
        system=SYSTEM_PROMPT,
    )

    if len(full_bytes) > SFT_MAX_LEN or not any(mask):
        continue

    seen.add(key)
    row["example_hash"] = key
    deduped.append(row)

def assign_split(example_hash: str):
    value = int(example_hash[:8], 16) / 0xFFFFFFFF
    if value < 0.02:
        return "test"
    if value < 0.04:
        return "val"
    return "train"

split_rows = {"train": [], "val": [], "test": []}
for row in deduped:
    split_rows[assign_split(row["example_hash"])].append(row)

for split, rows in split_rows.items():
    output = SFT_DIR / f"{split}.jsonl"
    with output.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

print("SFT examples:")
for split, rows in split_rows.items():
    kinds = {}
    for row in rows:
        kinds[row["kind"]] = kinds.get(row["kind"], 0) + 1
    print(f"  {split:5s}: {len(rows):6d}  {kinds}")


In [ ]:
# 10) Create a time-budgeted CUDA/AMP ForgeLoop SFT trainer.

import textwrap
from pathlib import Path

sft_trainer_path = Path("scripts/train_forgeloop_sft.py")

sft_trainer_code = r"""
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch

from cfna.chat import load_checkpoint
from cfna.training.dialogue_data import encode_example, make_sft_batch

def read_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

def amp_context(device, enabled):
    if enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def make_batch(rows, device, system, max_len):
    pairs = [(row["prompt"], row["response"]) for row in rows]
    batch = make_sft_batch(pairs, system=system, max_len=max_len)
    return {
        "byte_ids": torch.from_numpy(batch["byte_ids"]).to(device),
        "target_mask": torch.from_numpy(batch["target_mask"]).to(device),
    }

@torch.no_grad()
def evaluate(model, rows, device, system, max_len, amp_enabled, max_examples=64):
    model.eval()
    values = []
    for start in range(0, min(len(rows), max_examples), 4):
        chunk = rows[start:start + 4]
        if not chunk:
            continue
        batch = make_batch(chunk, device, system, max_len)
        if not batch["target_mask"].any():
            continue
        with amp_context(device, amp_enabled):
            logits, _ = model(batch["byte_ids"])
            loss = model.masked_token_loss(
                logits,
                batch["byte_ids"],
                batch["target_mask"],
            )
        values.append(loss.float().item())
    return float(np.mean(values)) if values else float("nan")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True)
    ap.add_argument("--train", required=True)
    ap.add_argument("--val", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--minutes", type=float, default=60)
    ap.add_argument("--batch", type=int, default=4)
    ap.add_argument("--max-len", type=int, default=384)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--log-every", type=int, default=25)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--system", required=True)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--no-amp", action="store_true")
    args = ap.parse_args()

    rng = np.random.default_rng(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp_enabled = device.type == "cuda" and not args.no_amp

    train_rows = read_jsonl(args.train)
    val_rows = read_jsonl(args.val)
    if not train_rows or not val_rows:
        raise ValueError("SFT train and validation files must both be non-empty.")

    model, base_checkpoint = load_checkpoint(args.base)
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    best_path = Path(str(out).replace(".pt", "") + "_best.pt")
    best_val = float("inf")
    evals_since_best = 0

    def _atomic_torch_save(payload, dest):
        tmp = Path(str(dest) + ".tmp")
        torch.save(payload, tmp)
        tmp.replace(dest)
    history = []
    sft_step = 0

    if args.resume and out.exists():
        checkpoint = torch.load(out, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["state_dict"])
        if "optimizer" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer"])
            for group in optimizer.param_groups:
                group["lr"] = args.lr
        history = list(checkpoint.get("sft_history", []))
        sft_step = int(checkpoint.get("sft_step", 0))
        print(f"resumed SFT from step {sft_step}")

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    print("device:", device)
    if device.type == "cuda":
        print("gpu:", torch.cuda.get_device_name(0))
    print("amp:", amp_enabled)
    print("train examples:", len(train_rows), "| val examples:", len(val_rows))
    print("model params:", f"{model.num_params():,}")

    start = time.time()
    run_steps = 0

    def save():
        torch.save(
            {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "config": base_checkpoint["config"],
                "step": base_checkpoint.get("step", 0),
                "history": base_checkpoint.get("history", []),
                "sft_step": sft_step,
                "sft_history": history,
                "sft_system": args.system,
                "sft_train_path": str(args.train),
            },
            out,
        )

    while (time.time() - start) < args.minutes * 60:
        indices = rng.integers(0, len(train_rows), size=args.batch)
        rows = [train_rows[int(index)] for index in indices]
        batch = make_batch(rows, device, args.system, args.max_len)

        if not batch["target_mask"].any():
            continue

        model.train()
        optimizer.zero_grad(set_to_none=True)

        with amp_context(device, amp_enabled):
            logits, _ = model(batch["byte_ids"])
            loss = model.masked_token_loss(
                logits,
                batch["byte_ids"],
                batch["target_mask"],
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        sft_step += 1
        run_steps += 1

        if run_steps % args.log_every == 0:
            val_loss = evaluate(
                model,
                val_rows,
                device,
                args.system,
                args.max_len,
                amp_enabled,
            )
            elapsed = (time.time() - start) / 60
            record = {
                "sft_step": sft_step,
                "run_step": run_steps,
                "train_loss": float(loss.float().item()),
                "val_loss": val_loss,
                "minutes": elapsed,
            }
            history.append(record)
            if val_loss < best_val:
                best_val = val_loss
                evals_since_best = 0
                _atomic_torch_save(
                    {
                        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                        "config": base_checkpoint["config"],
                        "sft_step": sft_step,
                        "sft_history": history,
                        "sft_system": args.system,
                        "best_val_loss": best_val,
                    },
                    best_path,
                )
                print(f"  new best val loss {best_val:.3f} -> {best_path}", flush=True)
            else:
                evals_since_best += 1
            print(
                f"sft step {sft_step:6d} | run {run_steps:5d} | {elapsed:6.1f}m | "
                f"train loss {record['train_loss']:.3f} | val loss {val_loss:.3f}",
                flush=True,
            )
            save()
            if evals_since_best >= 20:
                print(f"early stop: no val improvement for 20 evals (best {best_val:.3f})", flush=True)
                break

    save()
    print(f"saved SFT checkpoint -> {out} ({run_steps} new SFT steps)")

if __name__ == "__main__":
    main()
"""

sft_trainer_path.write_text(textwrap.dedent(sft_trainer_code).lstrip(), encoding="utf-8")
print("Wrote:", sft_trainer_path)


In [ ]:
# 11) Run ForgeLoop + general-assistant instruction tuning.

import os
import subprocess
import sys
from pathlib import Path

command = [
    sys.executable,
    "-u",
    "scripts/train_forgeloop_sft.py",
    "--base", str(LARGE_BASE_CKPT),
    "--train", str(SFT_DIR / "train.jsonl"),
    "--val", str(SFT_DIR / "val.jsonl"),
    "--out", str(SFT_CKPT),
    "--minutes", str(SFT_TRAIN_MINUTES),
    "--batch", str(SFT_BATCH_SIZE),
    "--max-len", str(SFT_MAX_LEN),
    "--lr", str(SFT_LEARNING_RATE),
    "--log-every", "25",
    "--system", SYSTEM_PROMPT,
]

if SFT_CKPT.exists():
    command.append("--resume")

env = os.environ.copy()
env["PYTHONPATH"] = str(Path.cwd()) + os.pathsep + env.get("PYTHONPATH", "")

print("Running:", " ".join(command))
process = subprocess.Popen(
    command,
    cwd=Path.cwd(),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"SFT failed with exit code {return_code}")

print("Instruction-tuned checkpoint:", SFT_CKPT)


In [ ]:
# 12) Test the instruction-tuned assistant.

from cfna.chat import Conversation, load_checkpoint

from pathlib import Path
_best = Path(str(SFT_CKPT).replace(".pt", "") + "_best.pt")
_load_from = _best if _best.exists() else SFT_CKPT
print("loading:", _load_from)
assistant_model, assistant_state = load_checkpoint(str(_load_from))
conversation = Conversation(
    assistant_model,
    system=assistant_state.get("sft_system", SYSTEM_PROMPT),
    temperature=0.2,
    max_new=180,
    min_new=12,
    max_ctx=384,
)

prompts = [
    "Hello CFNA. Introduce yourself in one sentence.",
    "A corpus manifest has training records but no validation records. Diagnose the failure.",
    (
        "Review a Python loader that reads JSONL records and crashes when the file is empty. "
        "Give a bounded ForgeLoop plan; do not claim you ran tests."
    ),
]

for prompt in prompts:
    reply = conversation.say(prompt)
    print("User:", prompt)
    print("Assistant:", reply)
    print()


In [ ]:
# 13) Optional: persist checkpoints and manifests to Google Drive.
# Set BACKUP_TO_DRIVE=True only after mounting permission is acceptable.

from pathlib import Path

CORPUS_DIR = Path("corpus_large")
LARGE_BASE_CKPT = Path("checkpoints/cfna_base_large.pt")
SFT_DIR = Path("data/cfna_forgeloop_sft")
SFT_CKPT = Path("checkpoints/cfna_forgeloop_sft.pt")

BACKUP_TO_DRIVE = True

if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    backup_dir = Path("/content/drive/MyDrive/CFNA_checkpoints")
    backup_dir.mkdir(parents=True, exist_ok=True)

    for path in [
        LARGE_BASE_CKPT,
        Path(str(LARGE_BASE_CKPT).replace(".pt", "") + "_best.pt"),
        Path(str(SFT_CKPT).replace(".pt", "") + "_best.pt"),
        Path("checkpoints/cfna_base_35m.pt"),
        Path("checkpoints/cfna_base_35m.pt.json"),
        Path(str(LARGE_BASE_CKPT) + ".json"),
        SFT_CKPT,
        CORPUS_DIR / "manifest.jsonl",
        SFT_DIR / "train.jsonl",
        SFT_DIR / "val.jsonl",
        SFT_DIR / "test.jsonl",
    ]:
        if path.exists():
            destination = backup_dir / path.name
            shutil.copy2(path, destination)
            print("Backed up:", destination)
else:
    print("Backup disabled. Set BACKUP_TO_DRIVE=True to enable.")
